# 04 Challenge

Merged notebook for the challenge module.

In [ ]:
# @title ⚙️ Environment Setup (Run this first!)
!pip install pymatgen numpy matplotlib scikit-learn -q
print("✅ Packages installed")

In [ ]:
# @title 📂 Load Tutorial Data
import os

REPO = "MRS_CH08_Tutorial"
REPO_URL = "https://github.com/Szymanski-Group/MRS_CH08_Tutorial.git"

if os.path.basename(os.getcwd()) == REPO:
    print("✅ Data already present")
elif os.path.exists(REPO):
    os.chdir(REPO)
    print("✅ Data already present")
else:
    !git clone {REPO_URL} -q
    os.chdir(REPO)
    print("✅ Data loaded successfully")


## 04a - Open Challenge

In [ ]:
import csv
from pathlib import Path

import matplotlib.pyplot as plt

from tutorial_utils.conventional import profile_correlation as pc
from tutorial_utils.sections import challenge_baseline as vis_ch


def create_challenge_baseline(
    mystery_dir="data/challenge/mystery_patterns",
    reference_dir="data/reference_structures",
    output_dir="outputs/conventional/profile_correlation",
    top_k_to_print=5,
    max_experiment_patterns=None,
    min_angle=10.0,
    max_angle=80.0,
    wavelength="CuKa",
    reference_intensity_threshold=1.0,
    baseline_percentile=5.0,
    fwhm=0.30,
    gauss_frac=0.2,
):
    """Step-by-step challenge baseline using profile correlation."""
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    vis_ch.OUTPUT_DIR = output_dir
    vis_ch.TOP_K_TO_PRINT = top_k_to_print
    vis_ch.FWHM = fwhm
    vis_ch.GAUSS_FRAC = gauss_frac

    mystery_files = sorted(Path(mystery_dir).glob("*.xy"))
    if max_experiment_patterns is not None:
        mystery_files = mystery_files[:max_experiment_patterns]

    ref_lib = pc.load_reference_stick_library(
        sorted(Path(reference_dir).glob("*.cif")),
        min_angle=min_angle,
        max_angle=max_angle,
        wavelength=wavelength,
        intensity_threshold=reference_intensity_threshold,
    )

    all_rows = []
    print("\n=== Challenge Baseline (Profile Correlation) ===")
    print(f"Mystery patterns: {len(mystery_files)}")
    print(f"Reference phases: {len(ref_lib)}")

    for mystery_file in mystery_files:
        pattern_name = mystery_file.stem
        two_theta, exp_profile = pc.load_experimental_profile(
            mystery_file,
            min_angle=min_angle,
            max_angle=max_angle,
            baseline_percentile=baseline_percentile,
        )

        by_pearson, by_cosine, simulated_profiles = pc.rank_phases(
            exp_profile,
            two_theta,
            ref_lib,
            fwhm=fwhm,
            gauss_frac=gauss_frac,
        )

        print(f"\n--- {pattern_name} ---")
        vis_ch.print_rank_table(pattern_name, by_pearson, "pearson", "Pearson")
        vis_ch.print_rank_table(pattern_name, by_cosine, "cosine", "Cosine")

        vis_ch.plot_summary(pattern_name, two_theta, exp_profile, by_pearson[0], by_cosine[0], simulated_profiles)

        for row in by_pearson:
            all_rows.append({"pattern": pattern_name, "phase": row["phase"], "pearson": row["pearson"], "cosine": row["cosine"]})

    csv_file = output_dir / "challenge_profile_correlations.csv"
    with open(csv_file, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["pattern", "phase", "pearson", "cosine"])
        writer.writeheader()
        writer.writerows(all_rows)

    print(f"\nSaved ranking table: {csv_file}")
    return all_rows


def get_challenge_predictions(
    mystery_dir="data/challenge/mystery_patterns",
    reference_dir="data/reference_structures",
    min_angle=10.0,
    max_angle=80.0,
    wavelength="CuKa",
    reference_intensity_threshold=1.0,
    baseline_percentile=5.0,
    fwhm=0.30,
    gauss_frac=0.2,
):
    """Return top-1 phase prediction for each mystery pattern."""
    ref_lib = pc.load_reference_stick_library(
        sorted(Path(reference_dir).glob("*.cif")),
        min_angle=min_angle,
        max_angle=max_angle,
        wavelength=wavelength,
        intensity_threshold=reference_intensity_threshold,
    )
    predictions = []
    for mystery_file in sorted(Path(mystery_dir).glob("*.xy")):
        two_theta, exp_profile = pc.load_experimental_profile(
            mystery_file,
            min_angle=min_angle,
            max_angle=max_angle,
            baseline_percentile=baseline_percentile,
        )
        by_pearson, _, _ = pc.rank_phases(exp_profile, two_theta, ref_lib, fwhm=fwhm, gauss_frac=gauss_frac)
        predictions.append((mystery_file.name, by_pearson[0]["phase"]))
    return predictions


# 04a — Open Challenge

Use the mystery patterns to test what you learned from the conventional and ML modules.

## Build a Baseline Solver
This cell applies full-profile correlation to each mystery pattern and records the top predicted phase.

In [ ]:
create_challenge_baseline()
predictions = get_challenge_predictions()
predictions[:5]


## Compare Against Ground Truth
Ground truth labels can contain phases outside the reference library, so this baseline is intentionally imperfect.

In [ ]:
import csv

truth_map = {}
with open("data/challenge/ground_truth.csv", newline="") as fh:
    for row in csv.DictReader(fh):
        truth_map[row["new_filename"]] = row["old_filename"].replace(".xy", "")
n_exact = 0
for fname, pred in predictions:
    true_label = truth_map.get(fname, "")
    exact = pred == true_label
    n_exact += int(exact)
    print(f"{fname}: predicted={pred:16s} | true={true_label}")
print(f"\nExact top-1 matches: {n_exact}/{len(predictions)}")


## 🧪 Try Your Own Strategy
Try combining multiple methods from earlier modules, adding your own preprocessing, or training a custom model for this challenge set.

## Summary
- The challenge set is designed to expose limitations of simple single-method pipelines.
- Missing reference phases make this a realistic open-set problem.
- Better performance usually comes from combining preprocessing, robust simulation, and model-based ranking.